In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%%writefile models.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

class AttentionDoubleLSTM(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, num_layers=2):
        super().__init__()
        self.projection = nn.Linear(input_dim, hidden_dim)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.1)
        self.attention_fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        e_t = F.relu(self.projection(x))
        lstm_out, _ = self.lstm(e_t)
        attn_scores = self.attention_fc(lstm_out)
        attn_weights = F.softmax(attn_scores, dim=1)
        c_ctx = torch.sum(attn_weights * lstm_out, dim=1)
        return c_ctx

class PPOAgent(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128):
        super().__init__()
        self.feature_extractor = AttentionDoubleLSTM(input_dim, hidden_dim)
        self.critic = nn.Linear(hidden_dim, 1)
        self.actor_targ = nn.Linear(hidden_dim, 4)
        self.actor_lr   = nn.Linear(hidden_dim, 3)
        self.actor_mult = nn.Linear(hidden_dim, 3)
        self.actor_enh  = nn.Linear(hidden_dim, 3)

    def forward(self, state):
        c_ctx = self.feature_extractor(state)
        value = self.critic(c_ctx)
        return value, [self.actor_targ(c_ctx), self.actor_lr(c_ctx), self.actor_mult(c_ctx), self.actor_enh(c_ctx)]

    def get_action(self, state, action=None):
        value, logits_list = self.forward(state)
        actions, log_probs, entropies = [], [], []
        for i, logits in enumerate(logits_list):
            dist = Categorical(logits=logits)
            act = dist.sample() if action is None else action[:, i]
            actions.append(act)
            log_probs.append(dist.log_prob(act))
            entropies.append(dist.entropy())
        return torch.stack(actions, dim=-1), torch.stack(log_probs, dim=-1).sum(dim=-1), entropies, value

Overwriting models.py


In [17]:
%%writefile inference.py
import torch
import numpy as np
from models import PPOAgent
from collections import deque

class RealTimeAutoscaler:
    def __init__(self, model_path="ppo_double_lstm_scaler.pth"):
        # 1. Khởi tạo lại đúng cấu trúc bộ não mạng Neural cũ
        self.agent = PPOAgent(input_dim=14, hidden_dim=128)

        # 2. Nạp toàn bộ trọng số đã được tối ưu hóa từ 1000 vòng train vào
        self.agent.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        self.agent.eval()  # Chuyển sang chế độ suy luận (Inference Mode), tắt dropout.

        # 3. Duy trì cửa sổ trượt w=3 để ghi nhớ lịch sử metrics
        self.state_buffer = deque(maxlen=3)
        for _ in range(3):
            self.state_buffer.append(np.zeros(14))

    def predict_next_action(self, live_metrics_14d):
        """
        Hàm nhận vào vector metrics 14 chiều lấy trực tiếp theo thời gian thực
        (Ví dụ trích xuất từ Prometheus API hoặc KVM Agent gộp về)
        """
        # Đẩy trượt thông số mới vào bộ đệm
        self.state_buffer.append(live_metrics_14d)

        # Khôi phục cấu trúc ma trận Tensor chuẩn (1, 3, 14)
        state_tensor = torch.FloatTensor(np.array(self.state_buffer)).unsqueeze(0)

        # Thực hiện suy luận không tính toán đạo hàm (Tiết kiệm tài nguyên tối đa)
        with torch.no_grad():
            action, _, _, _ = self.agent.get_action(state_tensor)

        # Giải mã hành động cấu hình hạ tầng
        a_targ, a_lr, a_mult, a_enh = action[0].tolist()

        target_mapping = {0: 30, 1: 50, 2: 70, 3: 90}
        recommended_cpu_target = target_mapping[a_targ]

        return recommended_cpu_target

# =========================================================
# VÒNG LẶP ĐIỀU KHIỂN HẠ TẦNG THỰC TẾ (CONTROL LOOP AGENT)
# =========================================================
if __name__ == "__main__":
    # Khởi tạo bộ điều phối cấu hình
    scaler = RealTimeAutoscaler(model_path="ppo_double_lstm_scaler.pth")

    print("=== BỘ ĐIỀU PHỐI AI ĐÃ SẴN SÀNG CHẠY TRÊN HỆ THỐNG PRODUCTION ===")

    # Giả lập vòng lặp quét thông số định kỳ hệ thống
    # Trong thực tế, đây sẽ là hàm while True: gọi API sau mỗi 5 hoặc 15 phút
    for step in range(1, 5):
        # 1. Đo đạc thông số thực tế từ máy chủ tại thời điểm hiện tại
        # Giả lập vector thu thập từ hệ thống giám sát
        live_sampled_metrics = np.random.uniform(0.1, 0.8, 14)

        # 2. Đưa vào cho mô hình AI dự đoán hành động cấu hình tối ưu
        action_decision = scaler.predict_next_action(live_sampled_metrics)

        # 3. Thực thi hành động xuống hạ tầng cụm node thông qua CLI/API
        print(f"[Nhịp {step}] Nhận thông số hạ tầng -> AI ra quyết định điều chỉnh Target CPU: {action_decision}%")
        # Chỗ này bạn sẽ chèn lệnh gọi API thực tế: os.system(f"openstack server set ...") hoặc k8s API

Writing inference.py


In [10]:
%%writefile environment.py
import numpy as np
import torch
import pandas as pd
from collections import deque
import math

class KubernetesEnv:
    def __init__(self, csv_path, max_steps=500):
        self.max_steps = max_steps
        self.df = pd.read_csv(csv_path)
        self.total_rows = len(self.df)
        self.current_row = 0
        self.state_buffer = deque(maxlen=3)
        self.cpu_target = 50

    def reset(self):
        self.current_step = 0
        if self.current_row + self.max_steps >= self.total_rows: self.current_row = 0
        for _ in range(3): self.state_buffer.append(np.zeros(14))
        return self._get_tensor_state()

    def _get_metrics_from_csv(self):
        row_data = self.df.iloc[self.current_row]
        self.current_row += 1
        s_t = np.zeros(14)
        s_t[1] = float(row_data['value'])
        s_t[3] = self.cpu_target / 100.0
        return s_t

    def _get_tensor_state(self):
        return torch.FloatTensor(np.array(self.state_buffer)).unsqueeze(0)

    def step(self, actions):
        self.current_step += 1
        a_targ = actions[0][0].item()
        self.cpu_target = {0: 30, 1: 50, 2: 70, 3: 90}[a_targ]
        new_s_t = self._get_metrics_from_csv()
        self.state_buffer.append(new_s_t)
        u_cpu = new_s_t[1] * 100.0
        diff = abs(u_cpu - self.cpu_target)
        reward = 1.0 if diff <= 10 else math.exp(-diff / 50.0)
        return self._get_tensor_state(), reward, self.current_step >= self.max_steps

Overwriting environment.py


In [15]:
%%writefile train.py
import torch
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from models import PPOAgent
from environment import KubernetesEnv

# HYPERPARAMETERS
LEARNING_RATE = 2e-4
GAMMA = 0.99
GAE_LAMBDA = 0.93
CLIP_EPSILON = 0.2
ENTROPY_COEFF = 0.01
UPDATE_EPOCHS = 10
BATCH_SIZE = 128
ROLLOUT_STEPS = 512
NUM_EPISODES = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hệ thống đang thực thi huấn luyện trên thiết bị: {device}")

env = KubernetesEnv(csv_path="/content/drive/MyDrive/Dataset/data.csv", max_steps=ROLLOUT_STEPS)
agent = PPOAgent(input_dim=14, hidden_dim=128).to(device)
optimizer = optim.Adam(agent.parameters(), lr=LEARNING_RATE)

def train():
    # Khởi tạo các mảng lưu lịch sử để phục vụ vẽ đồ thị báo cáo
    reward_history = []
    loss_history = []

    print("=== KHỞI CHẠY TIẾN TRÌNH HUẤN LUYỆN 1000 EPISODES ===")
    for episode in range(1, NUM_EPISODES + 1):
        state = env.reset().to(device)
        states, actions, log_probs, rewards, dones, values = [], [], [], [], [], []

        # PHA 1: THU THẬP TRẢI NGHIỆM
        for _ in range(ROLLOUT_STEPS):
            with torch.no_grad():
                action, log_prob, _, value = agent.get_action(state)

            next_state, reward, done = env.step(action)
            next_state = next_state.to(device)

            states.append(state)
            actions.append(action)
            log_probs.append(log_prob)
            rewards.append(reward)
            dones.append(done)
            values.append(value)

            state = next_state
            if done:
                break

        states = torch.cat(states, dim=0)
        actions = torch.cat(actions, dim=0)
        log_probs = torch.cat(log_probs, dim=0)
        values = torch.cat(values, dim=0).squeeze()

        # PHA 2: TÍNH TOÁN LỢI THẾ GAE
        returns = np.zeros_like(rewards)
        advantages = np.zeros_like(rewards)
        gae = 0
        next_value = 0 if dones[-1] else values[-1].item()

        for t in reversed(range(len(rewards))):
            delta = rewards[t] + GAMMA * next_value * (1 - dones[t]) - values[t].item()
            gae = delta + GAMMA * GAE_LAMBDA * (1 - dones[t]) * gae
            advantages[t] = gae
            returns[t] = advantages[t] + values[t].item()
            next_value = values[t].item()

        advantages = torch.FloatTensor(advantages).to(device)
        returns = torch.FloatTensor(returns).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # PHA 3: TỐI ƯU HÓA TRỌNG SỐ PPO
        epoch_losses = []
        for _ in range(UPDATE_EPOCHS):
            _, new_log_probs, entropies, new_values = agent.get_action(states, actions)
            new_values = new_values.squeeze()

            ratio = torch.exp(new_log_probs - log_probs)
            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1.0 - CLIP_EPSILON, 1.0 + CLIP_EPSILON) * advantages
            actor_loss = -torch.min(surr1, surr2).mean()

            critic_loss = F.mse_loss(new_values, returns)
            total_entropy = torch.stack(entropies).sum(dim=0).mean()
            loss = actor_loss + 0.5 * critic_loss - ENTROPY_COEFF * total_entropy

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())

        # Lưu lại giá trị trung bình của chu kỳ này
        reward_history.append(sum(rewards))
        loss_history.append(np.mean(epoch_losses))

        if episode % 10 == 0 or episode == 1:
            print(f"Episode {episode:04d}/{NUM_EPISODES} | Tích lũy Reward: {sum(rewards):.2f} | Tổng Loss: {np.mean(epoch_losses):.4f}")

    # ==========================================
    # PHẦN XUẤT FILE SAU KHI HOÀN THÀNH 1000 VÒNG
    # ==========================================
    print("\n=== HOÀN THÀNH HUẤN LUYỆN! ĐANG XUẤT TÀI NGUYÊN ===")

    # 1. Lưu cấu trúc trọng số mạng neural (Bộ não AI)
    torch.save(agent.state_dict(), "ppo_double_lstm_scaler.pth")
    print("-> Đã lưu file mô hình: ppo_double_lstm_scaler.pth")

    # 2. Vẽ và xuất biểu đồ xung lực Reward
    plt.figure(figsize=(10, 5))
    plt.plot(reward_history, color='#1f77b4', linewidth=1.5, label='Cumulative Reward')
    plt.title('Convergence Curve: Cumulative Reward Over Episodes', fontsize=12, fontweight='bold')
    plt.xlabel('Episode', fontsize=10)
    plt.ylabel('Total Reward', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.savefig('ppo_reward_convergence.pdf', bbox_inches='tight')  # Xuất đuôi .pdf chất lượng cao cho báo cáo
    plt.savefig('ppo_reward_convergence.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("-> Đã xuất đồ thị Reward: ppo_reward_convergence.pdf/.png")

    # 3. Vẽ và xuất biểu đồ hội tụ hàm Loss
    plt.figure(figsize=(10, 5))
    plt.plot(loss_history, color='#d62728', linewidth=1.5, label='Training Loss')
    plt.title('Model Optimization: Total PPO Loss Over Episodes', fontsize=12, fontweight='bold')
    plt.xlabel('Episode', fontsize=10)
    plt.ylabel('Loss Value', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()
    plt.savefig('ppo_loss_convergence.pdf', bbox_inches='tight')
    plt.savefig('ppo_loss_convergence.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("-> Đã xuất đồ thị Loss: ppo_loss_convergence.pdf/.png")

if __name__ == "__main__":
    train()

Overwriting train.py


In [13]:
# Kiểm tra danh sách file hiện có
!ls -l *.py

# Chạy huấn luyện nếu file đã tồn tại
import os
if os.path.exists('train.py'):
    !python train.py
else:
    print('Lỗi: Bạn chưa chạy các cell %%writefile ở trên để tạo file!')

-rw-r--r-- 1 root root 1459 Jun 22 08:55 environment.py
-rw-r--r-- 1 root root 1979 Jun 22 08:55 models.py
-rw-r--r-- 1 root root 5894 Jun 22 08:56 train.py
Hệ thống đang thực thi huấn luyện trên thiết bị: cuda
=== KHỞI CHẠY TIẾN TRÌNH HUẤN LUYỆN 1000 EPISODES ===
Episode 0001/20 | Tích lũy Reward: 206.08 | Tổng Loss: 12.3842
Episode 0010/20 | Tích lũy Reward: 249.63 | Tổng Loss: 11.4580
Episode 0020/20 | Tích lũy Reward: 418.10 | Tổng Loss: 37.2556

=== HOÀN THÀNH HUẤN LUYỆN! ĐANG XUẤT TÀI NGUYÊN ===
-> Đã lưu file mô hình: ppo_double_lstm_scaler.pth
-> Đã xuất đồ thị Reward: ppo_reward_convergence.pdf/.png
-> Đã xuất đồ thị Loss: ppo_loss_convergence.pdf/.png
